# Agents Laboratory — Constructing a Goal-Based Agent

**Artificial Intelligence — Laboratory Exercise**
Constructing a Goal-Based Agent for the Warehouse Navigation Problem.

This notebook works through every task in `Agents.pdf`:

1. **Task 1** — Understanding the problem (conceptual answers).
2. **Task 2** — Designing the agent (components + block diagram).
3. **Task 3** — Implementing and testing a goal-based agent in Python.

> The emphasis of the lab is *specifying an AI problem, designing an intelligent agent,
> and testing/validating the program* — not Python trivia. The code below is deliberately
> self-contained and documented so each design decision is visible.


## The Warehouse Map

```
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
```

* `S` — starting position (loading bay)
* `G` — goal (dispatch area)
* `#` — obstacle (shelving, cannot be crossed)
* `.` — free space

The vehicle may move **Up, Down, Left, Right**; one move changes its position by one grid square.


In [1]:
# The warehouse as a list of strings (one string per row).
WAREHOUSE = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################",
]

rows = len(WAREHOUSE)
cols = len(WAREHOUSE[0])
print(f"Grid size: {rows} rows x {cols} columns")
for r in WAREHOUSE:
    print(r)


Grid size: 7 rows x 21 columns
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


## Task 1 — Understanding the Problem

1. **What is the environment?**
   A fully-observable, static, discrete 2-D grid world. Each cell is either free (`.`), an
   obstacle (`#`), the start (`S`) or the goal (`G`). It is *deterministic*: a move always
   produces the intended adjacent cell (or is blocked by a wall).

2. **What is the goal of the agent?**
   To reach the goal cell `G` from the start cell `S` along a collision-free path (never
   entering a `#` cell and never leaving the grid).

3. **What actions are available to the agent?**
   Four movement actions — Up, Down, Left, Right — each moving one cell. An action is only
   valid if the destination cell is inside the grid and is not an obstacle.

4. **What information must the agent maintain to choose its next action?**
   Its current position, the set of already-visited positions (to avoid cycles / re-expansion),
   a frontier of positions still to explore, and the parent of each discovered position so the
   final path can be reconstructed.

5. **Why is this a goal-based agent rather than a simple reflex agent?**
   A simple reflex agent maps the current percept directly to an action via fixed rules and has
   no notion of *where it is trying to get to*. Here the agent must consider the **consequences**
   of a sequence of moves and select actions because they lead toward an explicit goal `G`.
   It plans a path using search — behaviour that depends on a goal, not just the current percept.

### Think About It — a warehouse twice as large
Uninformed breadth-first search still *works* (it remains complete and optimal for unit-cost
moves), but the number of states grows with the area, so time and memory grow roughly
quadratically with the linear size. On much larger maps an **informed** search such as A\* with a
Manhattan-distance heuristic becomes far more appropriate because it expands many fewer states.
Additional difficulties: larger branching of viable routes, memory pressure from the frontier
and visited set, and (in a real warehouse) dynamic obstacles / other vehicles that would break
the static-environment assumption.


## Task 2 — Designing the Agent

| Component | Design for this problem |
|---|---|
| **Environment** | 2-D grid parsed from the ASCII map; walls block movement. |
| **Current state** | `(row, col)` coordinate of the vehicle. |
| **Goal** | The coordinate of `G`; goal test is `state == goal`. |
| **Available actions** | Up `(-1,0)`, Down `(+1,0)`, Left `(0,-1)`, Right `(0,+1)`, each valid only if the target cell is in-bounds and not `#`. |
| **Decision-making component** | A **breadth-first search** planner that expands the frontier until `G` is found, then reconstructs the path via stored parents. |

### Block diagram (goal-based architecture)

```
             +-----------------------------------------------+
             |                   AGENT                        |
  percepts   |  +-----------+     +------------------------+  |
 ----------> |  |  State    | --> |  Decision maker        |  |
 (grid +     |  | (position,|     |  (BFS planner:         |  |   action
  current    |  |  frontier,|     |   expand frontier,     |  | --------->
  position)  |  |  visited, |     |   test goal, choose    |  |  (Up/Down/
             |  |  parents) | <-- |   next move toward G)  |  |  Left/Right)
             |  +-----------+     +-----------+------------+  |
             |                                |   ^          |
             |                        +-------v---+------+   |
             |                        |  Goal:  reach G   |   |
             |                        +-------------------+   |
             +-----------------------------------------------+
                               ^                     |
                               |     ENVIRONMENT     v
                        +---------------------------------+
                        |  Warehouse grid (static, known) |
                        +---------------------------------+
```

The agent perceives the grid and its position, maintains internal state, and the
decision-maker chooses actions **because they move it toward the goal** `G`.


## Task 3 — Implementation (the "LLM-generated" goal-based agent)

Rather than asking an external LLM to "write a program", we give a precise specification and
implement it here. The planner uses **breadth-first search (BFS)**.

**Why BFS?** Every move costs the same (one step), so BFS is *complete* (it always finds a path
if one exists) and *optimal* (the first time it reaches `G` it has found a path with the fewest
moves). It needs no heuristic, which keeps the agent simple and correct for this unit-cost grid.


In [2]:
from collections import deque

def parse_map(grid):
    """Return (start, goal, blocked-set, nrows, ncols) from an ASCII map."""
    start = goal = None
    blocked = set()
    for r, line in enumerate(grid):
        for c, ch in enumerate(line):
            if ch == "S":
                start = (r, c)
            elif ch == "G":
                goal = (r, c)
            elif ch == "#":
                blocked.add((r, c))
    return start, goal, blocked, len(grid), len(grid[0])


# The four actions, as (name, (d_row, d_col)).
ACTIONS = [("Up", (-1, 0)), ("Down", (1, 0)), ("Left", (0, -1)), ("Right", (0, 1))]


def goal_based_agent(grid):
    """Goal-based agent: breadth-first search from S to G.

    Returns a dict describing the outcome: path (list of coords), moves (action names),
    length, and number of states expanded.
    """
    start, goal, blocked, nrows, ncols = parse_map(grid)
    if start is None or goal is None:
        raise ValueError("Map must contain both S and G.")

    frontier = deque([start])          # positions still to explore (FIFO => BFS)
    came_from = {start: (None, None)}  # position -> (parent position, action taken)
    expanded = 0

    while frontier:
        current = frontier.popleft()
        expanded += 1

        if current == goal:                     # goal test
            # Reconstruct path by walking parents back to the start.
            path, moves = [], []
            node = current
            while node is not None:
                path.append(node)
                parent, action = came_from[node]
                if action is not None:
                    moves.append(action)
                node = parent
            path.reverse()
            moves.reverse()
            return {"found": True, "path": path, "moves": moves,
                    "length": len(path) - 1, "expanded": expanded,
                    "start": start, "goal": goal}

        r, c = current
        for name, (dr, dc) in ACTIONS:
            nxt = (r + dr, c + dc)
            nr, nc = nxt
            in_bounds = 0 <= nr < nrows and 0 <= nc < ncols
            if in_bounds and nxt not in blocked and nxt not in came_from:
                came_from[nxt] = (current, name)
                frontier.append(nxt)

    return {"found": False, "path": [], "moves": [], "length": None,
            "expanded": expanded, "start": start, "goal": goal}


In [3]:
def render(grid, path):
    """Return the map as a string with the path marked by '*' (S and G preserved)."""
    chars = [list(row) for row in grid]
    for (r, c) in path:
        if chars[r][c] not in ("S", "G"):
            chars[r][c] = "*"
    return "\n".join("".join(row) for row in chars)


result = goal_based_agent(WAREHOUSE)

if result["found"]:
    print(f"Path found!  Length = {result['length']} moves, "
          f"states expanded = {result['expanded']}")
    print("\nFirst moves:", result["moves"][:12], "...")
    print("\nPath drawn on the map ('*' = path):\n")
    print(render(WAREHOUSE, result["path"]))
else:
    print("No path exists from S to G.")


Path found!  Length = 20 moves, states expanded = 59

First moves: ['Right', 'Right', 'Right', 'Down', 'Right', 'Right', 'Right', 'Up', 'Right', 'Right', 'Right', 'Right'] ...

Path drawn on the map ('*' = path):

#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


In [4]:
# Full move sequence and the explicit coordinate path.
print("Action sequence from S to G:")
print(" -> ".join(result["moves"]))
print(f"\nTotal moves: {result['length']}")
print("\nCoordinate path (row, col):")
print(result["path"])


Action sequence from S to G:
Right -> Right -> Right -> Down -> Right -> Right -> Right -> Up -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right

Total moves: 20

Coordinate path (row, col):
[(1, 1), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (2, 6), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (1, 16), (1, 17), (1, 18), (1, 19)]


### Testing and validating the agent

A program that prints *a* path is not automatically correct. We validate it with checks whose
expected answers are known in advance.


In [5]:
def validate(grid, result):
    """Independently verify that the returned path is legal and reaches the goal."""
    start, goal, blocked, nrows, ncols = parse_map(grid)
    path = result["path"]
    assert path[0] == start, "Path must begin at S"
    assert path[-1] == goal, "Path must end at G"
    for (r, c) in path:
        assert (r, c) not in blocked, f"Path enters an obstacle at {(r, c)}"
        assert 0 <= r < nrows and 0 <= c < ncols, "Path leaves the grid"
    # Every consecutive pair must differ by exactly one orthogonal step.
    for (r1, c1), (r2, c2) in zip(path, path[1:]):
        assert abs(r1 - r2) + abs(c1 - c2) == 1, "Non-adjacent step in path"
    # Moves must match the coordinate transitions.
    assert len(result["moves"]) == len(path) - 1
    return True


print("Validation on the warehouse map:",
      "PASSED" if validate(WAREHOUSE, result) else "FAILED")

# --- Test: no-solution map (goal walled off) -> agent must report failure, not loop ---
NO_SOLUTION = [
    "#######",
    "#S....#",
    "###.###",
    "#...#G#",
    "#######",
]
ns = goal_based_agent(NO_SOLUTION)
print("No-solution map  -> found =", ns["found"],
      "(expected False), states expanded =", ns["expanded"])

# --- Test: trivial map (G next to S) -> one-step solution ---
TRIVIAL = ["####", "#SG#", "####"]
tv = goal_based_agent(TRIVIAL)
print("Trivial map      -> found =", tv["found"], ", moves =", tv["moves"],
      "(expected one 'Right')")


Validation on the warehouse map: PASSED
No-solution map  -> found = False (expected False), states expanded = 9
Trivial map      -> found = True , moves = ['Right'] (expected one 'Right')


## Task 3 — Evaluation Questions

**1. Did the "LLM" generate a working program on the first attempt?**
Treating the implementation above as the generated artefact: yes, it runs and returns a valid,
shortest path. In practice, LLM output often needs a round of fixing — common failures are
row/column (x/y) confusion, forgetting bounds checks, or marking a node visited at the wrong
time (which can produce non-shortest paths or missed solutions).

**2. If not, how can you improve your prompt?**
Give a precise specification: state the map format, the coordinate convention, the exact action
set, the requirement for a *shortest collision-free* path, the no-path behaviour, and ask it to
print both the path and the number of states expanded. Supplying concrete test cases (trivial,
no-solution) in the prompt makes the generated code far more likely to be correct.

**3. What search algorithm did the agent choose?**
Breadth-first search (an uninformed / blind search).

**4. Why is this algorithm appropriate here?**
All moves cost 1, so BFS is complete and returns a shortest path, with no heuristic required.
An LLM typically selects BFS (or A\*) for grid navigation because unit-cost shortest-path on a
grid is the textbook use case; BFS is the simplest choice that is guaranteed optimal here. For
much larger grids, A\* with the Manhattan-distance heuristic would expand fewer states while
still returning a shortest path (explored in the dedicated Search lab).


## Summary

* Formulated warehouse navigation as a **goal-based** agent problem (state = position,
  goal = reach `G`, actions = 4 moves).
* Designed the agent's components and their interaction (block diagram).
* Implemented a documented **BFS** planner, ran it on the supplied map, and **validated** it
  with trivial and no-solution test cases.
* Result: a shortest collision-free path was found on the warehouse map (see the executed
  output above for the exact move count and states expanded).

The agent selects actions *because they lead toward the goal* — the defining feature of a
goal-based agent.
